In [1]:
import pandas as pd
import numpy as np

# Load clean dataset and raw data
clean_df = pd.read_csv('hsls_clean.csv')
df_raw = pd.read_csv('hsls_17_student_pets_sr_v1_0.csv')

# Extract GED status from raw data
# X4EVERGED = 1 means student obtained GED or equivalent credential
ged_status = df_raw[['STU_ID', 'X4EVERGED']].copy()
print("X4EVERGED value counts:")
print(ged_status['X4EVERGED'].value_counts().sort_index())

# Merge onto clean dataset
clean_df = clean_df.merge(ged_status, on='STU_ID', how='left')

# Recode target variable
# Original: X4EVERDROP = 1 includes both traditional dropouts and GED completers
# New: X4EVERDROP_TRAD = 1 for traditional dropouts only, GED completers recoded to 0
clean_df['X4EVERDROP_TRAD'] = clean_df['X4EVERDROP'].copy()
ged_mask = (clean_df['X4EVERDROP'] == 1) & (clean_df['X4EVERGED'] == 1)
clean_df.loc[ged_mask, 'X4EVERDROP_TRAD'] = 0

# Verify
print(f"\nOriginal X4EVERDROP distribution:")
print(clean_df['X4EVERDROP'].value_counts().sort_index())
print(f"\nNew X4EVERDROP_TRAD distribution:")
print(clean_df['X4EVERDROP_TRAD'].value_counts().sort_index())
print(f"\nGED completers recoded to 0: {ged_mask.sum()}")

# Save
clean_df.to_csv('hsls_clean_trad.csv', index=False)
print(f"\nSaved: hsls_clean_trad.csv")
print(f"Shape: {clean_df.shape}")

X4EVERGED value counts:
X4EVERGED
0    22594
1      909
Name: count, dtype: int64

Original X4EVERDROP distribution:
X4EVERDROP
0    13458
1     2360
Name: count, dtype: int64

New X4EVERDROP_TRAD distribution:
X4EVERDROP_TRAD
0    14118
1     1700
Name: count, dtype: int64

GED completers recoded to 0: 660


/var/folders/zq/n2ytqrdj68x900tvyf0nfcyw0000gn/T/ipykernel_8119/1004502118.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clean_df['X4EVERDROP_TRAD'] = clean_df['X4EVERDROP'].copy()



Saved: hsls_clean_trad.csv
Shape: (15818, 915)


In [2]:
print(pd.crosstab(clean_df['X4EVERDROP'], clean_df['X4EVERGED'], margins=True))

X4EVERGED       0    1    All
X4EVERDROP                   
0           13454    4  13458
1            1700  660   2360
All         15154  664  15818
